# Filtro Hanning
***
Aplica el filtro Hanning a cada uno de los ficheros ya procesados por ReadOriginalData.py. De esta forma devuelve los mismos ficheros pero con las variables no necesarias eliminadas y además filtradas de forma que los perfiles son suavizados.

In [ ]:
# Packages for data manipulation
import xarray as xr
import pandas as pd
import numpy as np

# Packages for data representation
import matplotlib.pyplot as plt

# Package for interpolation
from scipy.interpolate import interp1d

# Packages for file manipulation
import os

# Ignore warnings
import warnings
warnings.filterwarnings(action='ignore', message='Mean of empty slice')

## Seleccionar Directorio
***

In [ ]:
#Select the file we want to work in

directorio_entrada  =          './Data/corrected_sections/A01'
directorio_salida   = './Data/corrected_sections_filtrado/A01'


## Ajustar filtro Hanning 
***

In [ ]:
# Create the Hanning Kernel
kernel_width_dbar = 40  # Wide of the kernel in dbar
spatial_resolution = 1  # Define the spatial resolution in dbar
kernel_size_points = int(kernel_width_dbar * spatial_resolution)
kernel = np.hanning(kernel_size_points)
kernel = kernel / kernel.sum()  # Normalization

#  Función completa
***

In [ ]:
def procesar_variable(ds, variable, kernel, presion_interp, num_perfiles):
                perfiles_procesados = np.full((num_perfiles, len(presion_interp)), np.nan)
                for perfil_num in range(num_perfiles):
                    datos = variable.values[perfil_num, :]
                    presion = ds.pressure.values[perfil_num, :]

                    # Delete NaN before interpolation
                    valid_indices = ~np.isnan(datos) & ~np.isnan(presion)
                    if not np.any(valid_indices):  # If no valid data
                        perfiles_procesados[perfil_num, :] = np.nan
                        continue

                    datos_validos = datos[valid_indices]
                    presion_valida = presion[valid_indices]

                    # Interpolate values at fixed pressure axis
                    interp_func = interp1d(presion_valida, datos_validos, bounds_error=False, fill_value=np.nan)
                    datos_interp = interp_func(presion_interp)

                    # Save the interpolated profile
                    perfiles_procesados[perfil_num, :] = datos_interp

                # Calculate the mean for each level
                media_por_nivel = np.nanmean(perfiles_procesados, axis=0)

                # Subtract the mean
                perfiles_sin_media = perfiles_procesados - media_por_nivel

                # Aplicate Hanning filter by convolution
                perfiles_filtrados = np.array([
                    np.convolve(kernel, perfil, mode='same') for perfil in perfiles_sin_media
                ])

                # Sum again the mean
                perfiles_finales = perfiles_filtrados + media_por_nivel
                return perfiles_finales

In [ ]:
def fn_filtro_hanning(directorio_entrada, directorio_salida):
    # Create the directory for save files
    if not os.path.exists(directorio_salida):
        os.makedirs(directorio_salida)

    # Iterate for every file
    for archivo in os.listdir(directorio_entrada):
        if archivo.endswith('.nc'):
            # Open the dataset
            ruta_archivo = os.path.join(directorio_entrada, archivo)
            ds = xr.open_dataset(ruta_archivo)

            # Create a matrix for save the proccesed profiles
            num_perfiles = ds.sizes['N_PROF']
            presion_interp = np.arange(0, 6501, 1)
            num_niveles = len(presion_interp)

            

            # Proccessing temperature
            if 'ctd_temperature' in ds.data_vars:
                variable_temperatura = ds.ctd_temperature
            elif 'ctd_temperature_68' in ds.data_vars:
                variable_temperatura = ds.ctd_temperature_68
            elif 'ctd_temperature_unk' in ds.data_vars:
                variable_temperatura = ds.ctd_temperature_unk
            else:
                variable_temperatura = None

            if variable_temperatura is not None:
                perfiles_finales_temperatura = procesar_variable(ds, variable_temperatura, kernel, presion_interp, num_perfiles)
                ds['ctd_temperature_filt'] = (('N_PROF', 'pressure_interp'), perfiles_finales_temperatura)

            # Proccessing oxigen
            if 'ctd_oxygen_ml_l' in ds.data_vars:
                variable_oxigeno = ds.ctd_oxygen_ml_l
            elif 'ctd_oxygen' in ds.data_vars:
                variable_oxigeno = ds.ctd_oxygen
            else:
                variable_oxigeno = None

            if variable_oxigeno is not None:
                perfiles_finales_oxigeno = procesar_variable(ds,variable_oxigeno, kernel, presion_interp, num_perfiles)
                ds['ctd_oxygen_filt'] = (('N_PROF', 'pressure_interp'), perfiles_finales_oxigeno)

            # Proccesing salinity
            if 'ctd_salinity' in ds.data_vars:
                perfiles_finales_salinidad = procesar_variable(ds, ds.ctd_salinity, kernel, presion_interp, num_perfiles)
                ds['ctd_salinity_filt'] = (('N_PROF', 'pressure_interp'), perfiles_finales_salinidad)

            # Drop variables without filter
            variables_a_eliminar = ['ctd_temperature', 'ctd_temperature_68', 'ctd_temperature_unk',
                                    'ctd_oxygen_ml_l', 'ctd_oxygen', 'ctd_salinity', 'pressure_qc',
                                    'ctd_oxygen_qc', 'ctd_oxygen_ml_l_qc', 'ctd_temperature_qc',
                                    'ctd_salinity_qc', 'ctd_temperature_unk_qc', 'ctd_temperature_68_qc',
                                    'profile_type', 'geometry_container', 'btm_depth', 'btm_depth_qc']
            for var in variables_a_eliminar:
                if var in ds.data_vars:
                    ds = ds.drop_vars(var)

            # Drop unwanted coordenates
            coordenadas_a_eliminar = ['sample', 'pressure']
            for coord in coordenadas_a_eliminar:
                if coord in ds.coords:
                    ds = ds.drop_vars(coord)

            # Save the new variables
            ds['pressure_interp'] = ('pressure_interp', presion_interp)

            # Save the file
            ruta_salida = os.path.join(directorio_salida, archivo)
            ds.to_netcdf(ruta_salida)

            # Close the dataset
            ds.close()

In [ ]:
# Try with a specific path
fn_filtro_hanning(directorio_entrada, directorio_salida)

In [ ]:
# Execute with all the directories and files
directorio_base_entrada = './Data/corrected_sections/'
directorio_base_salida = './Data/corrected_sections_filtrado/'

for carpeta in os.listdir(directorio_base_entrada):
    ruta_carpeta_entrada = os.path.join(directorio_base_entrada, carpeta)
    ruta_carpeta_salida = os.path.join(directorio_base_salida, carpeta)

    # Verify if it is a directory
    if os.path.isdir(ruta_carpeta_entrada):
        print(f"Correcting Section {carpeta}")
        fn_filtro_hanning(ruta_carpeta_entrada, ruta_carpeta_salida)
        print("Done!")